# Franchise Cost Analysis — 172 Brands from Real FDD Filings

This dataset contains structured cost data for 172 U.S. franchise brands, extracted from 2025 Franchise Disclosure Documents.

Source: [FranchiseVS](https://franchisevs.com) | [API](https://franchisevs.com/api/brands)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data/franchise-costs-2025.csv')
print(f'{len(df)} brands across {df["category"].nunique()} categories')
df.head()

## Investment ranges by category

In [ ]:
cat_stats = df.groupby('category').agg(
    count=('brand_name', 'count'),
    avg_inv_low=('investment_low', 'mean'),
    avg_inv_high=('investment_high', 'mean')
).sort_values('avg_inv_low')

fig, ax = plt.subplots(figsize=(12, 6))
y = range(len(cat_stats))
ax.barh(y, cat_stats['avg_inv_high'] - cat_stats['avg_inv_low'],
        left=cat_stats['avg_inv_low'], color='#7C3AED', alpha=0.7)
ax.set_yticks(y)
ax.set_yticklabels([f"{cat} ({n})" for cat, n in zip(cat_stats.index, cat_stats['count'])])
ax.set_xlabel('Initial Investment ($)')
ax.set_title('Average Investment Range by Category')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

## Cheapest franchises to open

In [ ]:
cheapest = df.nsmallest(15, 'investment_low')[[
    'brand_name', 'category', 'investment_low', 'investment_high',
    'royalty_value', 'health_score'
]].reset_index(drop=True)
cheapest['investment_low'] = cheapest['investment_low'].apply(lambda x: f'${x:,.0f}')
cheapest['investment_high'] = cheapest['investment_high'].apply(lambda x: f'${x:,.0f}')
cheapest

## Royalty rates vs. revenue

In [ ]:
has_rev = df[(df['has_item19'] == 'yes') & (df['median_gross_revenue'] > 0) & (df['royalty_value'] > 0)].copy()

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    has_rev['royalty_value'],
    has_rev['median_gross_revenue'],
    c=has_rev['health_score'],
    cmap='RdYlGn',
    s=60,
    alpha=0.7,
    edgecolors='white',
    linewidth=0.5
)
plt.colorbar(scatter, label='Health Score')
ax.set_xlabel('Royalty Rate (%)')
ax.set_ylabel('Median Gross Revenue ($)')
ax.set_title('Royalty Rate vs. Unit Revenue (brands disclosing Item 19)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

# Label a few outliers
for _, row in has_rev.nlargest(3, 'median_gross_revenue').iterrows():
    ax.annotate(row['brand_name'], (row['royalty_value'], row['median_gross_revenue']),
                fontsize=8, ha='left', va='bottom')

plt.tight_layout()
plt.show()

## Unit growth — who's expanding vs. shrinking

In [ ]:
growth = df[df['net_growth_rate_pct'].notna() & (df['net_growth_rate_pct'] != '')].copy()
growth['net_growth_rate_pct'] = pd.to_numeric(growth['net_growth_rate_pct'])

print(f"Growing (positive net change): {len(growth[growth['net_growth_rate_pct'] > 0])}")
print(f"Shrinking (negative net change): {len(growth[growth['net_growth_rate_pct'] < 0])}")
print(f"Flat: {len(growth[growth['net_growth_rate_pct'] == 0])}")
print()

print('Fastest growing:')
growth.nlargest(5, 'net_growth_rate_pct')[['brand_name', 'category', 'total_units', 'net_growth_rate_pct']]